# From Combine turns to TE scouting: an exploratory audit

## Research question
Do Combine turning measurements support useful tight-end distinctions and transfer to NFL separation? Can verified pressure targets and historical team context make the scouting evidence easier to interpret?

## Data and linkage
We reconcile 41 players labelled TE in both Combine and NFL position fields across the 2023–2025 cohorts. All 41 receive a unique NFL-to-GSIS identifier match. Draft year, overall pick and GSIS agree for 30 drafted players; 11 are undrafted or unresolved and are not assigned an invented drafting franchise.

Tracking coordinates generate approach speed, post-turn speed and smoothed acceleration around a sustained bend. The bend rule requires a direction change of at least 15 degrees sustained over three frame centres, consistent turn direction, progress between 15% and 85%, and approach speed of at least 2 yards/second. Sampling and displacement quality checks precede measurement. Acceleration is derived from coordinates and converted to metres/second squared; it is not a force measurement.

The existing transfer study links Combine measurements to nearest-defender distance at pass release. Its development sample contains 15 eligible TEs and 1,858 routes from 2023–2024. Context-only MAE is 0.306302 yards in pooled nested cross-validation and 0.321375 in temporal validation. A two-penalty Combine variant achieves 0.312751 and 0.321342 respectively. The temporal difference is negligible and pooled performance is worse. The earlier 2025 comparison also does not improve: baseline 0.255202 versus movement-augmented 0.256528 yards. These findings do not establish incremental predictive value from turning features. Later variants and the present extension are exploratory; 2025 has already been inspected and is no longer an untouched test set.

## Four operational archetype proxies
Reference quantiles use the 2023–2024 cohorts and are not selected to improve NFL outcomes. Labels are multi-label and provisional, not four mutually exclusive validated classes.

* Off-Script Seam Buster: approach and post-turn speeds both reach the reference 75th percentiles, with at least two valid attempts. Three players qualify. Scripted drills cannot demonstrate off-script decisions.
* Inline Leverage Anchor: weight reaches the reference 75th percentile and bench reaches the reference median. Five qualify, while 30 lack required measurements. This does not demonstrate blocking leverage.
* Hard-Brake Separator: at least two valid attempts and one turn of at least 30 degrees with derived acceleration at or below −5 m/s². Forty of 41 qualify. This rule fails the discrimination gate and must not be used as a useful classifier.
* Apex Catch Radius: wingspan and vertical reach the reference 75th percentiles. Four qualify and seven lack measurements. These are physical proxies, not observed catch radius.

Human film review and agreement between reviewers remain necessary for construct validation. Missing measurements are not negative labels.

## QB–TE pressure catch rates
We join FTN participation pressure flags, supplied publicly through nflverse, to NFL play-by-play and competition targets using game and play identifiers. We use `was_pressure`; blocker-level `pressure_allowed` is not substituted for quarterback pressure. All 702 eligible rookie targets match play-by-play, receiver identity, team and completion flags. Restricting to the original drafting franchise yields 684 known-pressure targets for 24 players and 47 QB–TE pairs, with no unknown-pressure targets in this filtered sample.

Only five pairs have at least ten pressured and ten clean targets. Below, each fraction is receptions/targets:

| Rookie TE / QB | Pressured | Clean |
|---|---:|---:|
| Ja'Tavion Sanders / B. Young, 2024 | 8/11 (72.7%) | 10/12 (83.3%) |
| Theo Johnson / D. Jones, 2024 | 6/12 (50.0%) | 15/20 (75.0%) |
| Harold Fannin / S. Sanders, 2025 | 11/19 (57.9%) | 18/28 (64.3%) |
| Gunnar Helm / C. Ward, 2025 | 9/10 (90.0%) | 35/45 (77.8%) |
| Oronde Gadsden / J. Herbert, 2025 | 17/25 (68.0%) | 28/38 (73.7%) |

The downloadable table supplies Wilson 95% intervals. These are descriptive comparisons with small denominators, not estimates of causal chemistry or evidence that a Combine archetype predicts pressure performance. Target selection and quarterback context remain confounders.

## Drafting-franchise outcomes and historical references
The original-franchise table contains 50 observed player-seasons with targets, receptions, yards and charted route context. A separate observation ledger preserves unobserved seasons as missing, rather than treating missing charting as zero production. Seasons since draft remain separate to avoid comparing unequal career lengths.

The fixed historical reference uses regular-season TE production for the 2019 KC, 2020 TB, 2021 LAR and 2022 KC Super Bowl winners: 16 TE player-seasons. These seasons precede the earliest prospect cohort. They describe championship-team TE rooms; they do not show that a physical profile causes championships.

For each 2023, 2024 and 2025 cohort, all 32 teams receive three prospects ranked only by height/weight similarity to the prior-season target-weighted TE room: 288 rows total. Historical standard deviations scale the distances. No post-draft outcomes enter similarity ordering. Championship-reference distance is supplied separately. This is a preliminary physical-comparison screen, not validated team fit or a draft recommendation. All rows explicitly retain `validated_fit=False`.

## What the work establishes
The strongest finding is measurement discipline: an apparently intuitive braking threshold identifies almost everybody, movement additions have not shown reliable separation improvement, and pressure evidence is limited to five adequately sampled pairs. Context, honest missingness and auditable crosswalks currently offer more dependable structure than a polished overall scouting score.

Next steps are blinded turn and film annotation, reviewer agreement, predeclared revised proxy rules, position-role opportunity controls, and validation on genuinely unseen cohorts. The deliverable is a reproducible exploratory submission draft; validated archetypes and validated team recommendations remain unfinished research.

## Reproduction and sources
Run the accompanying notebook with the competition data attached and internet enabled. It downloads public external sources, checks snapshot hashes, runs the extension and writes audit tables. It does not upload a submission. The prior separation-model results above are historical results from the separately supplied V2 notebook; the extension notebook does not rerun that model.

Competition data: [NFL Big Data Bowl 2027](https://www.kaggle.com/competitions/nfl-big-data-bowl-2027/data). External datasets: [nflverse releases](https://github.com/nflverse/nflverse-data/releases). Pressure definition: [participation dictionary](https://nflreadr.nflverse.com/articles/dictionary_participation.html). Pressure data attribution: FTN Data via nflverse, CC BY-SA 4.0; derived pressure tables retain that attribution and share-alike notice. Exact URLs, timestamps and SHA-256 hashes are supplied in the source manifest.


## Run the extension
Internet access is needed for public external files (~230 MB). Attach competition data; do not redistribute its raw files. Hash changes stop execution for review. The original V2 separation analysis is a separate notebook, not recomputed here.

In [ ]:
from pathlib import Path
import os,json,hashlib,urllib.request,warnings
from concurrent.futures import ThreadPoolExecutor
roots=[Path('/kaggle/input'),Path('/content')]
if os.environ.get('NFL_DATA_DIR'):
    data=Path(os.environ['NFL_DATA_DIR'])
else:
    candidates=[p.parent for root in roots if root.exists() for p in root.rglob('combine_tracking.csv') if (p.parent/'players.csv').exists()]
    if not candidates: raise FileNotFoundError('Attach the competition dataset or set NFL_DATA_DIR to the CSV directory.')
    if len(candidates)>1: raise ValueError('Multiple datasets found; set NFL_DATA_DIR explicitly.')
    data=candidates[0]
work=Path('/kaggle/working' if Path('/kaggle/working').exists() else '/content')
external=work/'te_external';external.mkdir(exist_ok=True)
out=work/'te_scouting_extension';out.mkdir(exist_ok=True)
os.environ.update(NFL_DATA_DIR=str(data),TE_EXTERNAL_DIR=str(external),TE_SCOUT_OUT=str(out))
MANIFEST=[{'file': 'draft_picks.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/draft_picks/draft_picks.csv', 'bytes': 1661916, 'sha256': 'cc35472c1b12b58ea9588e2174dfd92b3cd02cf6b4fc8d31cb9757a4ea3e5f9f', 'downloaded_utc': '2026-11-02T18:19:24.898912+00:00'}, {'file': 'player_stats_2021.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats_2021.csv', 'bytes': 1712336, 'sha256': '41ff0aafb569b5d47836168bdd755a8dcac4581d40dc42efe2f7821a341e903a', 'downloaded_utc': '2026-11-02T18:19:25.052210+00:00'}, {'file': 'player_stats_2022.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats_2022.csv', 'bytes': 1689929, 'sha256': '42f3a4da856546bd9606dde29eecf54e8dbbd6627264bbd8e153dd34ecd0e535', 'downloaded_utc': '2026-11-02T18:19:25.036428+00:00'}, {'file': 'player_stats_2023.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats_2023.csv', 'bytes': 1695570, 'sha256': 'fc731277c81c7ee25fb6af50cebd29a2adf3f8aadddeec1c485231aa9f98bdcc', 'downloaded_utc': '2026-11-02T18:19:25.117853+00:00'}, {'file': 'player_stats_2024.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats_2024.csv', 'bytes': 1674023, 'sha256': '4f1598c9a6ba396a52e256762a0d9148abd22cdf856f26fc8825fa1d1dc49631', 'downloaded_utc': '2026-11-02T18:19:25.025293+00:00'}, {'file': 'player_stats_2020.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats_2020.csv', 'bytes': 1636728, 'sha256': '44bfa428eae68eb45ee78fc6789e5409983e192606a75f678402cb0fe3a05ad5', 'downloaded_utc': '2026-11-02T18:19:25.341557+00:00'}, {'file': 'player_stats_2019.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/player_stats/player_stats_2019.csv', 'bytes': 1580008, 'sha256': 'b1338e5f364cd385f75aecf15fec69be02d075f5545749ecb4bcbc351e09718a', 'downloaded_utc': '2026-11-02T18:19:25.456592+00:00'}, {'file': 'roster_2019.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/rosters/roster_2019.csv', 'bytes': 1001234, 'sha256': 'b00259cfddf2146512569423275dc17e142d31ddb13bfba6b0b826c9d3d810da', 'downloaded_utc': '2026-11-02T18:19:25.431293+00:00'}, {'file': 'roster_2020.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/rosters/roster_2020.csv', 'bytes': 988000, 'sha256': 'dcf25143db7b54ac488d042b6c24142340effced1d6c8599b2084a9f412b3ed2', 'downloaded_utc': '2026-11-02T18:19:25.552737+00:00'}, {'file': 'roster_2021.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/rosters/roster_2021.csv', 'bytes': 953822, 'sha256': 'a95e57ebe1df25224a4d19ed43aaffc5c2097a8b369954178b8592537ff77d34', 'downloaded_utc': '2026-11-02T18:19:25.583114+00:00'}, {'file': 'roster_2022.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/rosters/roster_2022.csv', 'bytes': 980115, 'sha256': '864719595d14909c73bd14110adcdcfc304c6621de9ba34feafb83666fac43c9', 'downloaded_utc': '2026-11-02T18:19:26.258940+00:00'}, {'file': 'roster_2023.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/rosters/roster_2023.csv', 'bytes': 967354, 'sha256': 'f98283e6ef420babc597092b5483109a4570f9b5f63b7bfcce69643290960b6c', 'downloaded_utc': '2026-11-02T18:19:25.810200+00:00'}, {'file': 'roster_2024.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/rosters/roster_2024.csv', 'bytes': 1005701, 'sha256': '97721aa9092edb15c1357f2dac359dba72f63c81cc0e69408c65c229c91f50c1', 'downloaded_utc': '2026-11-02T18:19:25.823120+00:00'}, {'file': 'roster_2025.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/rosters/roster_2025.csv', 'bytes': 1010039, 'sha256': '531ee5de386037bb17de3e3e3d50f7b4ed08789526bbaae9876fdd3dfb4d04ad', 'downloaded_utc': '2026-11-02T18:19:25.894621+00:00'}, {'file': 'play_by_play_2023.csv.gz', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_2023.csv.gz', 'bytes': 19169807, 'sha256': '4649804ee0f0a40b41e51ec75a1ce921949d7fab5459213488656b92f78560e8', 'downloaded_utc': '2026-11-02T18:19:26.407113+00:00'}, {'file': 'play_by_play_2024.csv.gz', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_2024.csv.gz', 'bytes': 19362351, 'sha256': '23370d5d10f8104d80d46a1fc5e61f4f6f5a3263fe96fe2dd629913cfcb08c06', 'downloaded_utc': '2026-11-02T18:19:26.954104+00:00'}, {'file': 'play_by_play_2025.csv.gz', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_2025.csv.gz', 'bytes': 19105296, 'sha256': '2f135887790a013fd004e609e37096bb4816d5cc80b9f19122e1bad478961978', 'downloaded_utc': '2026-11-02T18:19:26.908905+00:00'}, {'file': 'games.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/schedules/games.csv', 'bytes': 2187844, 'sha256': '7bf3e1d9bcf0805fd0fdd1b95a5f016798a3705beebfea9240689c14a0270f8c', 'downloaded_utc': '2026-11-02T18:19:26.830019+00:00'}, {'file': 'pbp_participation_2023.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/pbp_participation/pbp_participation_2023.csv', 'bytes': 49967956, 'sha256': 'ad01aeb4045ee19a4f086ff38b52b14c8f427d3401e529c3078a4545921650a9', 'attribution': 'FTN Data via nflverse', 'license': 'CC-BY-SA 4.0', 'downloaded_utc': '2026-11-02T18:20:19.237596+00:00'}, {'file': 'pbp_participation_2024.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/pbp_participation/pbp_participation_2024.csv', 'bytes': 49688308, 'sha256': 'b1f436a98b2a7759eb4ed1181e072a35c2666f9aeb356a49c943d28d6be6b0b9', 'attribution': 'FTN Data via nflverse', 'license': 'CC-BY-SA 4.0', 'downloaded_utc': '2026-11-02T18:20:19.917703+00:00'}, {'file': 'pbp_participation_2025.csv', 'url': 'https://github.com/nflverse/nflverse-data/releases/download/pbp_participation/pbp_participation_2025.csv', 'bytes': 49094943, 'sha256': '59069adfee7b0f464befba8a5e8be331e523633cc6a7ab403d37bcbcdfbe66ac', 'attribution': 'FTN Data via nflverse', 'license': 'CC-BY-SA 4.0', 'downloaded_utc': '2026-11-02T18:20:18.247790+00:00'}]
def fetch(item):
    path=external/item['file']
    if not path.exists():
        with urllib.request.urlopen(item['url'],timeout=180) as response: content=response.read()
        path.write_bytes(content)
    digest=hashlib.sha256(path.read_bytes()).hexdigest()
    if digest!=item['sha256']: raise ValueError('Source snapshot changed: '+item['file']+'. Review and update the manifest explicitly before rerunning.')
    return item['file']
with ThreadPoolExecutor(max_workers=4) as pool: print(list(pool.map(fetch,MANIFEST)))
(out/'source_manifest.json').write_text(json.dumps(MANIFEST,indent=2))


In [ ]:
"""Research extension: operational TE proxies, pressure catch rates, draft-team outcomes,
pre-draft historical references and provisional physical-similarity screens.
No verified archetype, causal effect or validated 32-team fit score is claimed.
External pressure data attribution: FTN Data via nflverse, CC-BY-SA 4.0.
"""
from pathlib import Path
import os,json,math,hashlib
import numpy as np
import pandas as pd
D=Path(os.environ.get('NFL_DATA_DIR','/content/nfl-big-data-bowl-2027'))
E=Path(os.environ.get('TE_EXTERNAL_DIR','/content/te_external'))
O=Path(os.environ.get('TE_SCOUT_OUT','/content/te_scouting_extension'));O.mkdir(parents=True,exist_ok=True)
SPEC={'version':'TE scouting operational research v1','threshold_reference_cohorts':[2023,2024],
 'archetypes':'multi-label provisional proxies; all human_validated and archetype_validated false',
 'off_script':'scripted drill proxy only: route approach and post speed >= reference 75th percentiles; cannot measure off-script decisions',
 'inline':'mass >= reference 75th percentile and bench >= reference median; cannot measure leverage/block integrity',
 'hard_brake':'at least 2 valid selected route attempts; at least one turn >=30deg with derived smoothed acceleration <=-5 m/s^2',
 'apex':'wingspan and vertical >= reference 75th percentiles; cannot measure actual catch radius',
 'pressure':'FTN via nflverse was_pressure; never substitute blocker pressure_allowed or qb_hit',
 'H1_window':'rookie regular season, original drafting franchise; matched non-nullified target plays; unknown pressure excluded and counted',
 'pair_minimum':'at least 10 pressured and 10 clean-pocket targets for primary descriptive pair-rate comparison',
 'catch':'Kaggle target true and pass_result=C versus C/I/IN eligible targets',
 'champion_reference':'2019-2022 Super Bowl winners, REG TE production; fixed before earliest 2023 cohort',
 'fit':'prior-season roster TE target-weighted height/weight; descriptive similarity shortlist only, not validated fit',
 'fit_candidates':'3 per team per draft cohort, no post-draft performance used in similarity; historical outcomes displayed separately',
 'selection':'same existing cohorts already explored; new analysis is retrospective exploratory, not new heldout validation'}
(O/'protocol.json').write_text(json.dumps(SPEC,indent=2))
ABBR={'LA':'LAR','LVR':'LV','OAK':'LV','GNB':'GB','KAN':'KC','NWE':'NE','NOR':'NO','SFO':'SF','TAM':'TB','JAC':'JAX','SD':'LAC','STL':'LAR'}
def team(s):return s.replace(ABBR) if isinstance(s,pd.Series) else ABBR.get(s,s)
def save(df,name):df.to_csv(O/(name+'.csv'),index=False)
def flag(v):return v.astype('string').str.lower().map({'true':True,'false':False,'1':True,'0':False})
players=pd.read_csv(D/'players.csv');combine=pd.read_csv(D/'combine_results.csv');tracking=pd.read_csv(D/'combine_tracking.csv',dtype={'event_id':'string'})
base=combine[combine.combine_position.eq('TE')].merge(players,on=['nfl_id','draft_year'],validate='one_to_one');save(base[['nfl_id','display_name','draft_year','nfl_position']],'TE_position_audit')
base=base[base.nfl_position.eq('TE')].copy();assert len(base)==41
# Stable numeric NFL id -> GSIS link from year-specific external rosters.
rosters=[]
for year in range(2019,2026):
 r=pd.read_csv(E/f'roster_{year}.csv');r['team']=team(r.team);rosters.append(r)
R=pd.concat(rosters,ignore_index=True)
links=R[R.gsis_it_id.notna()][['gsis_it_id','gsis_id']].drop_duplicates();links.gsis_it_id=links.gsis_it_id.astype(int)
links=links[links.gsis_it_id.isin(base.nfl_id)&links.gsis_id.notna()]
if links.gsis_it_id.duplicated().any():raise ValueError('Ambiguous NFL->GSIS link')
base=base.merge(links.rename(columns={'gsis_it_id':'nfl_id'}),on='nfl_id',how='left',validate='one_to_one')
draft=pd.read_csv(E/'draft_picks.csv');draft['draft_team']=team(draft.team)
C=draft[['season','pick','draft_team','gsis_id','pfr_player_name']].rename(columns={'season':'draft_year','pick':'draft_overall_pick','gsis_id':'draft_gsis_id'})
assert not C.duplicated(['draft_year','draft_overall_pick']).any()
base=base.merge(C,on=['draft_year','draft_overall_pick'],how='left',validate='many_to_one')
base['draft_status']=np.where(base.draft_overall_pick.isna(),'undrafted_original_signing_team_unresolved',np.where(base.draft_team.notna(),'drafted_verified','draft_crosswalk_unresolved'))
check=base.draft_status.eq('drafted_verified');assert base.loc[check,'gsis_id'].notna().all();assert base.loc[check,'gsis_id'].eq(base.loc[check,'draft_gsis_id']).all()
save(base[['nfl_id','display_name','draft_year','draft_overall_pick','gsis_id','draft_team','pfr_player_name','draft_status']],'draft_team_crosswalk')
# Tracking-derived first sustained bend. Coordinates yards, time seconds, derivatives converted to m/s².
T=tracking[tracking.nfl_id.isin(base.nfl_id)&tracking.entity_type.eq('PLAYER')&tracking.drill_name.isin(['HOOK_ROUTE_RIGHT','IN_ROUTE_LEFT'])].copy()
T['timestamp']=pd.to_datetime(T.time,utc=True);KEY=['nfl_id','draft_year','event_id','drill_type','drill_name','attempt'];assert not T.duplicated(KEY+['timestamp']).any()
rows=[]
for key,g in T.groupby(KEY,sort=True):
 g=g.sort_values('timestamp');xy=g[['x','y']].to_numpy(float);ns=g.timestamp.astype('int64').to_numpy();dt=np.diff(ns)/1e9;tt=(ns-ns[0])/1e9;n=len(g)
 s=np.r_[np.nan,np.linalg.norm(np.diff(xy,axis=0),axis=1)/dt];good=np.r_[False,(abs(dt-.1)<=.015)&(s[1:]>=0)&(s[1:]<=15)]
 path=np.r_[0,np.cumsum(np.linalg.norm(np.diff(xy,axis=0),axis=1))];progress=path/path[-1] if path[-1]>0 else path
 smooth=pd.Series(s).rolling(3,center=True,min_periods=3).mean().to_numpy();acc=np.r_[np.nan,np.diff(smooth)/dt]*.9144
 candidates=[]
 for j in range(8,n-8):
  if not .15<=progress[j]<=.85 or not good[j-7:j+9].all():continue
  v=xy[j]-xy[j-3];w=xy[j+3]-xy[j]
  if min(np.linalg.norm(v),np.linalg.norm(w))<.3:continue
  angle=np.degrees(np.arctan2(v[0]*w[1]-v[1]*w[0],v@w));approach=float(np.median(s[j-6:j-2]))
  if approach<2 or abs(angle)<15:continue
  candidates.append(dict(j=j,angle=angle,approach=approach,through=float(min(np.mean(s[k-1:k+2]) for k in range(j-2,j+3))),post=float(np.median(s[j+3:j+7])),brake_m_s2=float(np.nanmin(acc[j-4:j+5]))))
 episodes=[]
 for c in candidates:
  if not episodes or c['j']!=episodes[-1][-1]['j']+1 or np.sign(c['angle'])!=np.sign(episodes[-1][-1]['angle']):episodes.append([])
  episodes[-1].append(c)
 eligible=[ep for ep in episodes if len(ep)>=3];r=dict(zip(KEY,key));r['status']='valid' if eligible else 'no_sustained_episode'
 if eligible:
  c=max(eligible[0],key=lambda x:abs(x['angle']));r.update(c);r['seconds']=float(tt[c['j']]);r['retention']=c['through']/c['approach'];r['brake_turn_evidence']=abs(c['angle'])>=30 and c['brake_m_s2']<=-5
 rows.append(r)
attempts=pd.DataFrame(rows);save(attempts,'archetype_attempt_features')
v=attempts[attempts.status.eq('valid')]
f=v.groupby('nfl_id').agg(route_attempts=('nfl_id','size'),route_drills=('drill_name','nunique'),approach_speed=('approach','median'),post_speed=('post','median'),speed_retention=('retention','median'),brake_turn_attempts=('brake_turn_evidence','sum'),min_brake_m_s2=('brake_m_s2','min')).reset_index()
base=base.merge(f,on='nfl_id',how='left',validate='one_to_one');ref=base[base.draft_year.isin([2023,2024])]
thresholds={c:float(ref[c].quantile(q)) for c,q in [('approach_speed',.75),('post_speed',.75),('combine_weight',.75),('bench_reps',.5),('wing_span',.75),('vertical',.75)]}
(O/'archetype_thresholds.json').write_text(json.dumps(thresholds,indent=2))
labelrows=[]
for _,r in base.iterrows():
 conditions=[('Off-Script Seam Buster','scripted_route_speed_proxy',['approach_speed','post_speed','route_attempts'],r.approach_speed>=thresholds['approach_speed'] and r.post_speed>=thresholds['post_speed'] and r.route_attempts>=2,'Scripted drill cannot establish off-script decision making.'),
 ('Inline Leverage Anchor','size_strength_proxy',['combine_weight','bench_reps'],r.combine_weight>=thresholds['combine_weight'] and r.bench_reps>=thresholds['bench_reps'],'Body mass and bench do not establish leverage, force or block integrity.'),
 ('Hard-Brake Separator','braking_turn_proxy',['route_attempts','brake_turn_attempts'],r.route_attempts>=2 and r.brake_turn_attempts>=1,'Threshold and movement phase are operational; NFL separation translation remains unproven.'),
 ('Apex Catch Radius','reach_jump_proxy',['wing_span','vertical'],r.wing_span>=thresholds['wing_span'] and r.vertical>=thresholds['vertical'],'Wingspan and jump are proxies, not measured catch radius.')]
 for archetype,proxy,needed,passed,limit in conditions:
  status='insufficient_measurements' if any(pd.isna(r[c]) for c in needed) else ('provisional_candidate' if passed else 'below_proxy_rule')
  labelrows.append(dict(nfl_id=r.nfl_id,display_name=r.display_name,draft_year=r.draft_year,archetype=archetype,proxy=proxy,status=status,human_validated=False,archetype_validated=False,limit=limit))
labels=pd.DataFrame(labelrows);save(labels,'TE_archetype_proxies');save(base,'TE_feature_table');save(labels.groupby(['archetype','status']).size().rename('players').reset_index(),'archetype_coverage')
readiness=[]
for archetype,g in labels[labels.draft_year.isin([2023,2024])].groupby('archetype'):
 observed=g.status.ne('insufficient_measurements');rate=float(g.loc[observed,'status'].eq('provisional_candidate').mean()) if observed.any() else np.nan
 readiness.append(dict(archetype=archetype,reference_players=len(g),measured_players=int(observed.sum()),reference_candidate_rate=rate,classification_ready=False,decision='reject_as_discriminative_classifier' if rate>=.9 else 'proxy_only_requires_construct_and_film_validation'))
save(pd.DataFrame(readiness),'archetype_readiness')
# Match complete passing data and true QB-pressure labels by game/play; preserve missingness.
P=[];U=[]
for year in [2023,2024,2025]:
 cols=['old_game_id','game_id','play_id','season','season_type','posteam','passer_player_id','passer_player_name','receiver_player_id','complete_pass','pass_attempt','qb_spike','qb_hit','first_down_pass']
 p=pd.read_csv(E/f'play_by_play_{year}.csv.gz',usecols=cols,low_memory=False);p['old_game_id']=pd.to_numeric(p.old_game_id,errors='coerce').astype('Int64');p['posteam']=team(p.posteam);P.append(p)
 u=pd.read_csv(E/f'pbp_participation_{year}.csv',usecols=['old_game_id','nflverse_game_id','play_id','was_pressure']);u['old_game_id']=pd.to_numeric(u.old_game_id,errors='coerce').astype('Int64');u['was_pressure']=flag(u.was_pressure);U.append(u)
pbp=pd.concat(P,ignore_index=True);pressure=pd.concat(U,ignore_index=True)
assert not pbp.duplicated(['old_game_id','play_id']).any();assert not pressure.duplicated(['old_game_id','play_id']).any()
pbp=pbp.merge(pressure[['old_game_id','play_id','was_pressure']],on=['old_game_id','play_id'],how='left',validate='one_to_one')
q=pd.read_csv(D/'player_play.csv',low_memory=False);q=q[q.nfl_id.isin(base.nfl_id)].copy();assert not q.duplicated(['game_id','play_id','nfl_id']).any()
games=pd.read_csv(D/'games.csv');q=q.merge(games[['game_id','season','season_type']],on='game_id',validate='many_to_one')
q=q.merge(base[['nfl_id','display_name','draft_year','draft_team','draft_status','gsis_id']],on='nfl_id',validate='many_to_one');q['team_abbr']=team(q.team_abbr)
lookup=pbp.drop(columns=['season','season_type']).rename(columns={'old_game_id':'game_id','game_id':'nflverse_game_id'})
q=q.merge(lookup,on=['game_id','play_id'],how='left',validate='many_to_one',indicator='pbp_match')
q['original_franchise']=q.draft_status.eq('drafted_verified')&q.team_abbr.eq(q.draft_team)
q['rookie']=q.season.eq(q.draft_year)&q.season_type.eq('REG')
q['eligible_target']=flag(q.target).eq(True)&q.pass_result.isin(['C','I','IN'])&q.play_nullified_by_penalty.eq('N')
q['reception']=q.pass_result.eq('C');q['receiver_conflict']=q.receiver_player_id.notna()&q.receiver_player_id.ne(q.gsis_id)
all_targets=q[q.rookie&q.eligible_target].copy();
assert all_targets.pbp_match.eq('both').all()
assert all_targets.team_abbr.eq(all_targets.posteam).all()
assert all_targets.reception.eq(all_targets.complete_pass.eq(1)).all()
save(all_targets[['nfl_id','display_name','draft_year','game_id','play_id','team_abbr','draft_team','passer_player_id','passer_player_name','pass_result','was_pressure','original_franchise','receiver_conflict','pbp_match']],'H1_target_link_audit')
primary=all_targets[all_targets.original_franchise&~all_targets.receiver_conflict&all_targets.pbp_match.eq('both')&all_targets.passer_player_id.notna()].copy()
def wilson(k,n):
 if not n:return (np.nan,np.nan)
 z=1.95996398454;p=k/n;den=1+z*z/n;mid=(p+z*z/(2*n))/den;rad=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/den;return mid-rad,mid+rad
pairs=[]
for key,g in primary.groupby(['nfl_id','display_name','draft_year','draft_team','passer_player_id','passer_player_name'],dropna=False):
 r=dict(zip(['nfl_id','display_name','draft_year','draft_team','qb_gsis_id','qb_name'],key));r['unknown_pressure_targets']=int(g.was_pressure.isna().sum())
 for val,name in [(True,'pressured'),(False,'clean')]:
  s=g[g.was_pressure.eq(val)];n=len(s);k=int(s.reception.sum());lo,hi=wilson(k,n);r.update({name+'_targets':n,name+'_receptions':k,name+'_catch_rate':k/n if n else np.nan,name+'_CI_low':lo,name+'_CI_high':hi})
 r['pressure_resilience']=r['pressured_catch_rate']-r['clean_catch_rate'];r['primary_pair_eligible']=r['pressured_targets']>=10 and r['clean_targets']>=10;pairs.append(r)
pairs=pd.DataFrame(pairs);save(pairs,'QB_TE_pressure_catch_rates')
coverage=all_targets.groupby('draft_year').agg(targets=('nfl_id','size'),players=('nfl_id','nunique'),pressure_known=('was_pressure','count'),receiver_conflicts=('receiver_conflict','sum')).reset_index();save(coverage,'H1_coverage')
# First-franchise outcomes by season: no trades or cumulative career conflation.
first=q[q.original_franchise&q.season_type.eq('REG')&q.play_nullified_by_penalty.eq('N')].copy();outcomes=[]
for key,g in first.groupby(['nfl_id','display_name','draft_year','draft_team','season']):
 tar=g[g.eligible_target];rt=g[g.route_ran.notna()&g.pass_result.isin(['C','I','IN'])];r=dict(zip(['nfl_id','display_name','draft_year','draft_team','season'],key));r.update(years_since_draft=int(key[-1]-key[2]),charted_games=g.game_id.nunique(),charted_player_plays=len(g),charted_routes=len(rt),targets=len(tar),receptions=int(tar.reception.sum()),receiving_yards=float(tar.rec_yards.sum()),yards_after_catch=float(tar.yards_after_catch.sum()),mean_separation=float(rt.separation_at_pass_forward.mean()),separation_routes=int(rt.separation_at_pass_forward.notna().sum()),catch_rate=float(tar.reception.mean()) if len(tar) else np.nan,targets_per_charted_route=len(tar)/len(rt) if len(rt) else np.nan);outcomes.append(r)
outcomes=pd.DataFrame(outcomes);save(outcomes,'first_drafting_team_season_outcomes')
ledger=[]
for _,r in base[base.draft_status.eq('drafted_verified')].iterrows():
 for year in range(int(r.draft_year),2026):
  present=((outcomes.nfl_id==r.nfl_id)&(outcomes.season==year)).any()
  ledger.append(dict(nfl_id=r.nfl_id,display_name=r.display_name,draft_year=r.draft_year,draft_team=r.draft_team,season=year,observed_original_team_rows=bool(present),status='observed' if present else 'no_charted_original_team_rows_not_zero_performance'))
save(pd.DataFrame(ledger),'first_team_observation_ledger')
# Source-derived champions before 2023, and TE room production from 2019-2022 REG.
sched=pd.read_csv(E/'games.csv');sb=sched[sched.game_type.eq('SB')&sched.season.between(2019,2022)].copy();sb['champion']=team(pd.Series(np.where(sb.home_score>sb.away_score,sb.home_team,sb.away_team),index=sb.index));save(sb[['season','game_id','champion','home_team','away_team','home_score','away_score']],'Super_Bowl_reference_seasons')
assert len(sb)==4 and sb.season.max()<2023
stats=[]
for y in range(2019,2025):
 x=pd.read_csv(E/f'player_stats_{y}.csv');x=x[x.season_type.eq('REG')&x.position.eq('TE')].copy();x['team']=team(x.recent_team);stats.append(x)
S=pd.concat(stats,ignore_index=True)
prod=S.groupby(['season','team','player_id','player_display_name'],as_index=False).agg(games_with_stats=('week','nunique'),targets=('targets','sum'),receptions=('receptions','sum'),receiving_yards=('receiving_yards','sum'),receiving_tds=('receiving_tds','sum'))
phys=R[R.position.eq('TE')][['season','team','gsis_id','height','weight']].drop_duplicates()
if phys.duplicated(['season','team','gsis_id']).any():raise ValueError('Conflicting roster body measurements')
prod=prod.merge(phys,left_on=['season','team','player_id'],right_on=['season','team','gsis_id'],how='left',validate='one_to_one')
champ=prod.merge(sb[['season','champion']],left_on=['season','team'],right_on=['season','champion'],validate='many_to_one');save(champ,'Super_Bowl_TE_room_reference')
# Team profiles use the previous season only. No future NFL prospect outcome enters screening.
profiles=[]
for (y,tm),g in prod[prod.season.between(2022,2024)].groupby(['season','team']):
 g=g[g.targets.gt(0)&g.height.notna()&g.weight.notna()]
 if not len(g):continue
 profiles.append(dict(draft_year=y+1,team=tm,reference_season=y,TEs=len(g),TE_targets=int(g.targets.sum()),profile_height=float(np.average(g.height,weights=g.targets)),profile_weight=float(np.average(g.weight,weights=g.targets))))
profiles=pd.DataFrame(profiles);assert profiles.reference_season.lt(profiles.draft_year).all();save(profiles,'pre_draft_team_TE_profiles')
std_h=max(float(R[R.position.eq('TE')&R.season.between(2019,2022)].height.std()),1.);std_w=max(float(R[R.position.eq('TE')&R.season.between(2019,2022)].weight.std()),10.)
known=champ[champ.targets.gt(0)&champ.height.notna()&champ.weight.notna()];ch=float(np.average(known.height,weights=known.targets));cw=float(np.average(known.weight,weights=known.targets))
label_strings=labels[labels.status.eq('provisional_candidate')].groupby('nfl_id').archetype.agg('; '.join)
short=[]
for _,pr in profiles.iterrows():
 candidates=base[base.draft_year.eq(pr.draft_year)&base.combine_height.notna()&base.combine_weight.notna()].copy()
 candidates['physical_distance']=np.hypot((candidates.combine_height-pr.profile_height)/std_h,(candidates.combine_weight-pr.profile_weight)/std_w)
 for rank,(_,c) in enumerate(candidates.sort_values(['physical_distance','nfl_id']).head(3).iterrows(),1):
  short.append(dict(draft_year=int(pr.draft_year),team=pr.team,similarity_order=rank,nfl_id=int(c.nfl_id),prospect=c.display_name,profile_season=int(pr.reference_season),height=c.combine_height,weight=c.combine_weight,physical_distance=float(c.physical_distance),champion_reference_distance=float(np.hypot((c.combine_height-ch)/std_h,(c.combine_weight-cw)/std_w)),archetype_proxy_candidates=label_strings.get(c.nfl_id,''),actual_draft_team=c.draft_team,validated_fit=False,evidence_status='INSUFFICIENT_FOR_SCOUTING_RECOMMENDATION',use='provisional two-trait similarity screen for human review; not a validated overall team-fit ranking'))
short=pd.DataFrame(short);save(short,'all_32_teams_provisional_shortlists')
coverage32=profiles.groupby('draft_year').team.nunique();assert coverage32.eq(32).all();assert len(short)==32*3*3;assert not short.validated_fit.any()
status=dict(TE_players=len(base),verified_drafted_players=int(check.sum()),undrafted_or_unresolved=int((~check).sum()),GSIS_matched_players=int(base.gsis_id.notna().sum()),archetype_status=labels.groupby('status').size().to_dict(),H1_pairs=len(pairs),H1_primary_pairs=int(pairs.primary_pair_eligible.sum()),H1_pressure_known_targets=int(primary.was_pressure.notna().sum()),H1_unknown_pressure_targets=int(primary.was_pressure.isna().sum()),H1_primary_players=int(primary.nfl_id.nunique()),first_team_player_seasons=len(outcomes),champion_seasons=len(sb),champion_TE_player_seasons=len(champ),teams_per_cohort={str(k):int(v) for k,v in coverage32.items()},shortlist_rows=len(short),archetypes_validated=False,fit_validated=False,submission_uploaded=False)
(O/'status.json').write_text(json.dumps(status,indent=2,default=int))
(O/'validation_checks.txt').write_text('PASS: 41 reconciled TEs; unique roster NFL->GSIS mapping; draft pick/year/GSIS agreement; unique game/play/player and external pressure keys; first-franchise filter; no future seasons in team or champion profiles; fixed 2023-24 proxy thresholds; unknown pressure retained separately; 32 teams per cohort and 288 screen rows; no validated archetype or fit claim.\n')
print(json.dumps(status,indent=2,default=int));print(labels.groupby(['archetype','status']).size().to_string());print(pairs[pairs.primary_pair_eligible][['display_name','draft_year','qb_name','pressured_targets','pressured_catch_rate','clean_targets','clean_catch_rate']].to_string(index=False))


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
P=Path(os.environ['TE_SCOUT_OUT'])
q=pd.read_csv(P/'QB_TE_pressure_catch_rates.csv');q=q[q.primary_pair_eligible].reset_index(drop=True)
fig,ax=plt.subplots(figsize=(10,6));y=np.arange(len(q))
for offset,prefix,color,label in [(-.13,'pressured','#d45d43','Pressured'),(.13,'clean','#286a9b','Clean')]:
 rate=q[prefix+'_catch_rate'];lo=q[prefix+'_CI_low'];hi=q[prefix+'_CI_high']
 ax.errorbar(rate,y+offset,xerr=[rate-lo,hi-rate],fmt='o',capsize=4,color=color,label=label)
 for i,r in q.iterrows():ax.text(1.02,i+offset,f"{r[prefix+'_receptions']}/{r[prefix+'_targets']}",va='center',fontsize=9)
ax.set_yticks(y,[f"{r.display_name} / {r.qb_name} ({r.draft_year})" for _,r in q.iterrows()]);ax.set_xlim(0,1);ax.set_xlabel('Catch rate with Wilson 95% interval');ax.invert_yaxis();ax.grid(axis='x',alpha=.2);ax.legend(loc='lower left');ax.set_title('Five QB–TE pairs: descriptive rates, small samples');fig.tight_layout();fig.savefig(P/'pressure_catch_rates.png',dpi=170,bbox_inches='tight');plt.close(fig)
c=pd.read_csv(P/'archetype_coverage.csv').pivot(index='archetype',columns='status',values='players').fillna(0)
fig,ax=plt.subplots(figsize=(10,5));left=np.zeros(len(c))
for key,color in [('provisional_candidate','#337ea5'),('below_proxy_rule','#b8c5ce'),('insufficient_measurements','#e5b45a')]:
 ax.barh(c.index,c[key],left=left,label=key.replace('_',' '),color=color);left+=c[key].to_numpy()
ax.set_xlabel('Players (41 total per proxy)');ax.set_title('Archetype proxies — none validated; Hard-Brake rule rejected');ax.legend(loc='upper center',bbox_to_anchor=(.5,-.12),ncol=3);fig.tight_layout();fig.savefig(P/'archetype_coverage.png',dpi=170,bbox_inches='tight');plt.close(fig)
display(__import__('IPython').display.Image(filename=str(P/'pressure_catch_rates.png')))
display(__import__('IPython').display.Image(filename=str(P/'archetype_coverage.png')))


In [ ]:
display(pd.read_csv(O/'archetype_readiness.csv'))
display(pd.read_csv(O/'H1_coverage.csv'))
print((O/'validation_checks.txt').read_text())
print((O/'status.json').read_text())